In [ ]:
%pip install pandas chromadb sentence-transformers


In [ ]:
import pandas as pd

faqs = pd.read_csv("faqs.csv")

print("Number of FAQs:", len(faqs))
display(faqs.head())

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).split())


faqs["question"] = faqs["question"].apply(clean_text)
faqs["answer"] = faqs["answer"].apply(clean_text)

faqs = faqs[
    (faqs["question"] != "") &
    (faqs["answer"] != "")
].drop_duplicates(subset=["question"])

faqs = faqs.reset_index(drop=True)

print("FAQs after cleaning:", len(faqs))
display(faqs.head())

In [ ]:
documents = []
metadatas = []
ids = []

for index, row in faqs.iterrows():
    text = (
        f"Question: {row['question']}\n"
        f"Answer: {row['answer']}"
    )

    documents.append(text)

    metadatas.append({
        "question": row["question"],
        "answer": row["answer"],
    })

    ids.append(f"faq-{index}")

print("Documents prepared:", len(documents))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

print("Embeddings generated:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="python_faqs",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

print("FAQs indexed:", collection.count())

In [ ]:
def search_faqs(query, top_k=3):
    query = clean_text(query)

    if not query:
        print("Please enter a question.")
        return

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count()),
        include=["metadatas", "distances"],
    )

    for rank, metadata in enumerate(
        results["metadatas"][0], start=1
    ):
        distance = results["distances"][0][rank - 1]

        print(f"\nMatch {rank}")
        print(f"Similarity: {1 - distance:.3f}")
        print("Question:", metadata["question"])
        print("Answer:", metadata["answer"])


search_faqs("How can I store a value in my program?")

In [ ]:
%pip install pandas chromadb sentence-transformers

In [ ]:
import pandas as pd

faqs = pd.read_csv("faqs.csv")

print("Number of FAQs:", len(faqs))
display(faqs.head())import pandas as pd

faqs = pd.read_csv("faqs.csv")

print("Number of FAQs:", len(faqs))
display(faqs.head())

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).split())


faqs["question"] = faqs["question"].apply(clean_text)
faqs["answer"] = faqs["answer"].apply(clean_text)

faqs = faqs[
    (faqs["question"] != "") &
    (faqs["answer"] != "")
].drop_duplicates(subset=["question"])

faqs = faqs.reset_index(drop=True)

print("FAQs after cleaning:", len(faqs))
display(faqs.head())

In [ ]:
documents = []
metadatas = []
ids = []

for index, row in faqs.iterrows():
    text = (
        f"Question: {row['question']}\n"
        f"Answer: {row['answer']}"
    )

    documents.append(text)

    metadatas.append({
        "question": row["question"],
        "answer": row["answer"],
    })

    ids.append(f"faq-{index}")

print("Documents prepared:", len(documents))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

print("Embeddings generated:", len(embeddings))
print("Vector dimensions:", len(embeddings[0]))

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="python_faqs",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

print("FAQs indexed:", collection.count())

In [ ]:
def search_faqs(query, top_k=3):
    query = clean_text(query)

    if not query:
        print("Please enter a question.")
        return

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count()),
        include=["metadatas", "distances"],
    )

    for rank, metadata in enumerate(
        results["metadatas"][0], start=1
    ):
        distance = results["distances"][0][rank - 1]

        print(f"\nMatch {rank}")
        print(f"Similarity: {1 - distance:.3f}")
        print("Question:", metadata["question"])
        print("Answer:", metadata["answer"])


search_faqs("How can I store a value in my program?")